# Lab 8 – K-Nearest Neighbors (KNN)
**Dataset:** Medical Insurance Cost (`insurance.csv`)  
**Target:** `smoker` — Binary classification (yes = 1 / no = 0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv('insurance.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(3)

## 2. Feature Preparation

In [ ]:
df_model = df.copy()
df_model['sex_enc'] = (df_model['sex'] == 'male').astype(int)
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)
df_model.drop(columns=['sex'], inplace=True)

X = df_model.drop(columns=['smoker'])
y = (df_model['smoker'] == 'yes').astype(int)
print('Features:', X.columns.tolist())

## 3. Standardize the Features

In [ ]:
scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('After scaling (first 5 rows):')
X_scaled.head().round(3)

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)

print(f'Training size : {X_train.shape}')
print(f'Test size     : {X_test.shape}')

## 5. Initial KNN Model (K = 1)

In [ ]:
knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X_train, y_train)
pred1 = knn1.predict(X_test)

print('=== K=1 Classification Report ===')
print(classification_report(y_test, pred1, target_names=['Non-Smoker','Smoker']))

## 6. Elbow Method — Find Best K

In [ ]:
error_rate = []
k_range    = range(1, 41)

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, y_train)
    error_rate.append(np.mean(knn_k.predict(X_test) != y_test))

plt.figure(figsize=(11, 5))
plt.plot(k_range, error_rate, color='royalblue', linestyle='--',
         marker='o', markerfacecolor='tomato', markersize=7)
plt.title('Error Rate vs K Value (Elbow Method)')
plt.xlabel('K'); plt.ylabel('Error Rate')
plt.xticks(range(1, 41, 2))
plt.tight_layout(); plt.show()

best_k = list(k_range)[error_rate.index(min(error_rate))]
print(f'Best K : {best_k}  (error rate = {min(error_rate):.4f})')

## 7. Retrain with Best K

In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
pred_best = knn_best.predict(X_test)

print(f'=== KNN (K={best_k}) — Confusion Matrix ===')
cm = confusion_matrix(y_test, pred_best)
print(cm)
print()
print(f'=== KNN (K={best_k}) — Classification Report ===')
print(classification_report(y_test, pred_best, target_names=['Non-Smoker','Smoker']))

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-Smoker','Smoker'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix — KNN (K={best_k})')
plt.tight_layout(); plt.show()

In [ ]:
acc_k1   = 1 - error_rate[0]
acc_best = 1 - min(error_rate)
print(f'Accuracy at K=1:      {acc_k1:.4f}')
print(f'Accuracy at K={best_k}: {acc_best:.4f}')
print(f'Improvement:          {acc_best - acc_k1:+.4f}')

**Interpretation:**  
At K=1 the model overfits — it memorises the training data and performs poorly on unseen points. The Elbow Method reveals the optimal K where the error rate stabilises, balancing bias and variance. Because `charges` separates smokers and non-smokers so cleanly in this dataset, KNN achieves high accuracy once features are standardised (without scaling, the large `charges` values would dominate all distance calculations).